# Fact Set Minifigs — Gold Layer

Fact table at the grain of one row per set + minifigure. Answers which minifigs come in which sets and how many.

## What this notebook does

1. **Read** — Loads sets, inventories, inventory minifigs, minifigs, and theme hierarchy from silver/gold.
2. **Transform** — Joins the chain: sets → inventories → inventory_minifigs → minifigs, enriched with theme names.
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/gold/delta_volume/fct_set_minifigs`.
4. **Register** — Creates the catalog table `lego_brickbase.gold.fct_set_minifigs`.

## Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col


SILVER_SETS              = "lego_brickbase.silver.foundation_sets"
SILVER_INVENTORIES       = "lego_brickbase.silver.foundation_inventories"
SILVER_INVENTORY_MINIFIGS = "lego_brickbase.silver.foundation_inventory_minifigs"
SILVER_MINIFIGS          = "lego_brickbase.silver.foundation_minifigs"
GOLD_THEME_HIERARCHY     = "lego_brickbase.gold.dim_theme_hierarchy"
DELTA_TABLE_PATH         = "/Volumes/lego_brickbase/gold/delta_volume/fct_set_minifigs"
CATALOG_TABLE            = "lego_brickbase.gold.fct_set_minifigs"

## Read and Transform

Join the full minifig inventory chain and enrich with set and theme details.

In [ ]:
df_sets         = spark.table(SILVER_SETS)
df_inv          = spark.table(SILVER_INVENTORIES)
df_inv_minifigs = spark.table(SILVER_INVENTORY_MINIFIGS)
df_minifigs     = spark.table(SILVER_MINIFIGS)
df_themes       = spark.table(GOLD_THEME_HIERARCHY)

df_fct = (
    df_inv_minifigs
    .join(df_inv, df_inv_minifigs["inventory_id"] == df_inv["inventory_id"], "inner")
    .join(df_sets, df_inv["set_key"] == df_sets["set_key"], "inner")
    .join(df_minifigs, df_inv_minifigs["minifig_key"] == df_minifigs["minifig_key"], "left")
    .join(df_themes, df_sets["theme_key"] == df_themes["theme_key"], "left")
    .select(
        df_sets["set_key"],
        df_inv_minifigs["minifig_key"],
        df_sets["set_number"],
        df_sets["set_name"],
        df_sets["year"],
        df_themes["theme_name"],
        df_themes["root_theme_name"],
        df_minifigs["minifig_set_number"],
        df_minifigs["minifigs_name"],
        df_minifigs["number_of_parts"].alias("minifig_number_of_parts"),
        df_inv_minifigs["quantity"],
        df_minifigs["minifig_set_image_url"],
    )
)

df_fct.printSchema()
df_fct.display(10, truncate=False)

## Write to Delta Volume, Register Catalog Table, and Apply Constraints

In [ ]:
(
    df_fct
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE} (
        set_key                  STRING  NOT NULL COMMENT 'Foreign key to dim_set; identifies the parent set',
        minifig_key              STRING  NOT NULL COMMENT 'Surrogate key identifying the minifigure',
        set_number               STRING           COMMENT 'Official LEGO set number (e.g. 75192-1)',
        set_name                 STRING           COMMENT 'Official name of the LEGO set',
        year                     INTEGER          COMMENT 'Year the set was released',
        theme_name               STRING           COMMENT 'Name of the direct theme the set belongs to',
        root_theme_name          STRING           COMMENT 'Name of the top-level ancestor theme',
        minifig_set_number       STRING           COMMENT 'Rebrickable catalog number assigned to this minifigure (e.g. fig-000001)',
        minifigs_name            STRING           COMMENT 'Descriptive name of the minifigure',
        minifig_number_of_parts  INTEGER          COMMENT 'Number of individual parts that make up this minifigure',
        quantity                 INTEGER          COMMENT 'Number of copies of this minifigure included in the set',
        minifig_set_image_url    STRING           COMMENT 'URL to the image of this minifigure',
        CONSTRAINT fct_set_minifigs_pk
            PRIMARY KEY (set_key, minifig_key),
        CONSTRAINT fct_set_minifigs_fk_set
            FOREIGN KEY (set_key) REFERENCES lego_brickbase.gold.dim_set (set_key)
    )
    COMMENT 'Fact table at the grain of one row per set + minifigure. Captures which minifigs are included in each set and how many, enriched with set and theme details.'
""")

spark.sql(f"""
    INSERT INTO {CATALOG_TABLE}
    SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")

print(f"Catalog table ready with constraints: {CATALOG_TABLE}")